# PHYS 602 — Scientific Python Programming

This notebook covers the tools that practicing physicists use daily:  
**NumPy** for fast array computing, **Numba** for JIT compilation, and **SciPy** for mathematical algorithms (optimization, curve fitting, integration, ODEs).  
A new **Advanced Matplotlib** section introduces multi-panel figures, histograms, error bars, and 2-D plots.

Work through each section sequentially. Exercise cells are marked.

---
> **  AI Autocomplete Notice**  
> If you are using VS Code, JupyterLab, or any editor with **GitHub Copilot** (or similar AI tools) enabled, please **turn it off** for these exercises.  
> Copilot will autocomplete exercise blanks before you have a chance to think through the answer yourself, which defeats the purpose of the exercises.  
>  
> *How to disable Copilot in VS Code:* click the Copilot icon in the status bar, then choose **Disable Globally** (or **Disable for Notebook**).  
---

In [ ]:
# Install required packages (run once)
#   Numba does NOT support Python 3.13 yet. If you get an
#    llvmlite error, revert to Python 3.11 or 3.12 first.
!pip install numpy numba matplotlib scipy line_profiler


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy as sp
print("All packages imported successfully!")


# 1  NumPy — Numerical Python

NumPy provides the **`ndarray`** (n-dimensional array), the cornerstone of scientific computing in Python. Under the hood, NumPy's core routines are written in C and Fortran, so operations run *orders of magnitude* faster than equivalent Python loops.

**Why NumPy?**  
* Vectorised operations eliminate slow Python `for` loops.  
* A rich library of mathematical, statistical, and linear algebra functions.  
* Memory-efficient (all elements share the same C-level type).  
* The foundation on which SciPy, Pandas, and TensorFlow are built.


## 1.1  Creating Arrays

```python
np.array([1, 2, 3])          # from a list
np.zeros(5)                  # 5 zeros
np.ones((3, 3))              # 3×3 matrix of ones
np.linspace(0, 1, 101)       # 101 points uniformly spaced 0–1
np.arange(0, 10, 2)          # [0, 2, 4, 6, 8] (like range)
np.logspace(0, 3, 50)        # 50 points logarithmically spaced 10⁰–10³
np.random.seed(42)
np.random.randn(100)         # 100 standard-normal random numbers
```

**dtype:** All elements share a single C-level type (e.g. `float64`, `int32`, `bool`). Mixing types forces **upcasting** — integers promote to floats, everything promotes to strings.


In [ ]:
# ── Example 1.1 — Creating Arrays ────────────────────────────────
import numpy as np

# From a Python list — dtype is inferred
a = np.array([1, 2, 3])
print("From list:", a, "| dtype:", a.dtype)

# Pre-filled arrays
print("zeros(4):", np.zeros(4))
print("ones((2,3)):\n", np.ones((2, 3)))

# Range-like constructors
print("arange(0,10,2):", np.arange(0, 10, 2))      # step-based
print("linspace(0,1,5):", np.linspace(0, 1, 5))     # count-based, inclusive
print("logspace(0,3,4):", np.logspace(0, 3, 4))     # 10^0 to 10^3

# Upcasting — all elements share one dtype
mixed = np.array([1, 2.5, True])
print("int+float+bool:", mixed.dtype, mixed)        # float64

In [ ]:
# ── Exercise 1.1 — Creating Arrays ──────────────────────────────
import numpy as np

# (a) Print the sum of [0,1,2,...,9] using np.sum and the .sum() method
arr = np.array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9])
print(    )   # YOUR CODE HERE — np.sum(arr)
print(    )   # YOUR CODE HERE — arr.sum()

# (b) What happens when mixed types are combined?
arr_mixed = np.array([0, 1.2, True, "hello"])
print(arr_mixed)          # observe the dtype
print(arr_mixed.dtype)

# (c) Create an array of booleans from the mixed list below
arr_vals = np.array([1.2, 1.4, 0, 0, 8.9, None, True, "False"])
arr_bool = arr_vals.astype(    )   # YOUR CODE HERE
print(arr_bool)

# (d) NEW — Create and print:
#     • 10 evenly spaced floats from 0 to 1
#     • 5 log-spaced points from 10^0 to 10^4
x_lin =    # YOUR CODE HERE
x_log =    # YOUR CODE HERE
print("Linear:", x_lin)
print("Log:   ", x_log)


## 1.2  Performance: List vs Array Appending

NumPy arrays have a **fixed size** — `np.append` creates a *new* array each time, requiring memory reallocation. The standard Python pattern of appending to a list and **converting once** is often faster for dynamic growth.

> Rule of thumb: if you know the size in advance, pre-allocate with `np.zeros(n)`. If you must grow dynamically, use a Python list and convert at the end.


In [ ]:
# ── Example 1.2 — Why np.append is slow ──────────────────────────
import numpy as np

# np.append ALWAYS creates a brand-new array (copies all existing data)
a = np.array([1, 2, 3])
b = np.append(a, 4)    # a is unchanged; b is a new object
print("original a:", a)
print("new b:     ", b)

# This means inside a loop, every iteration copies the growing array:
#   iteration 1 -> copy 1 element
#   iteration 2 -> copy 2 elements
#   ...
#   iteration n -> copy n elements
#   Total copies: 1+2+...+n = n(n+1)/2   -> O(n²) !
#
# Python list.append is O(1) amortised (doubles capacity when full),
# so build a list and convert ONCE at the end for O(n) total cost.

In [ ]:
# ── Exercise 1.2 — Appending Performance ────────────────────────
from time import time

large_n = int(1e4)

# Method 1: np.append inside a loop
tstart = time()
my_arr = np.array([])
for i in range(large_n):
    my_arr = np.append(my_arr, i)
print(f"np.append loop : {time()-tstart:.3f} s")

# Method 2: Python list + single conversion
tstart = time()
my_list = []
for i in range(large_n):
    my_list.append(i)
my_arr2 = np.array(my_list)
print(f"list + convert : {time()-tstart:.3f} s")

# Which is faster and why?


## 1.3  Math Operations with NumPy

NumPy functions apply **element-wise** to entire arrays:

| Function | Description |
|----------|-------------|
| `np.sin`, `np.cos`, `np.tan` | Trigonometric |
| `np.exp`, `np.log`, `np.log10` | Exponential / logarithm |
| `np.sqrt`, `np.abs`, `np.power` | Root / absolute / power |
| `np.mean`, `np.std`, `np.median` | Statistics |
| `np.max`, `np.min`, `np.argmax`, `np.argmin` | Extrema |
| `np.dot`, `np.matmul` | Matrix multiplication |
| `np.linalg.inv`, `np.linalg.det` | Linear algebra |

The `math` module offers the same functions but only for **scalars**. Calling `math.sin` on a NumPy array raises a `TypeError`.


In [ ]:
# ── Example 1.3 — NumPy vs math for array operations ─────────────
import math, numpy as np

# math functions work on scalars only
print("math.sin(1.0):", math.sin(1.0))

# np functions apply element-wise to entire arrays
x = np.array([0, np.pi/4, np.pi/2, np.pi])
print("np.sin(x):", np.sin(x))  # [0, 0.707, 1.0, ~0]

# Calling math.sin on an array raises TypeError:
try:
    math.sin(x)
except TypeError as e:
    print("math.sin on array:", e)

# Common element-wise operations
data = np.array([1.0, 4.0, 9.0, 16.0])
print("sqrt:", np.sqrt(data))
print("log: ", np.log(data))
print("mean:", np.mean(data), " std:", np.std(data))

### Timing Code with `%%timeit` and `%timeit`

Jupyter provides two built-in timing tools:

| Magic | Scope | Usage |
|-------|-------|-------|
| `%%timeit` | Entire cell | Put on the **first line** of the cell; times everything in it |
| `%timeit` | Single expression | Put at the start of the line to time |
| `%time` | Single expression (one run) | Like `%timeit` but runs once — good for slow operations |

`%%timeit` runs the cell many times automatically and reports **mean ± std dev**, e.g.:  
`2.1 ms ± 30 µs per loop (mean ± std. dev. of 7 runs, 100 loops each)`

> **Gotcha:** because `%%timeit` re-runs the cell, avoid defining state you rely on elsewhere (like fitting results) inside a `%%timeit` cell — put setup code *above* it instead.

The two cells below time the same computation — iterating `math.sin` in a Python loop vs applying `np.sin` to the whole array at once. Run both to see the speedup.

In [ ]:
# ── Exercise 1.3 — Math Operations ──────────────────────────────
import math

x_scalar = 1.2

# (a) Compare math.sin vs np.sin for a single value
y_math  =    # YOUR CODE HERE — math.sin
y_numpy =    # YOUR CODE HERE — np.sin
print(f"math.sin:  {y_math}")
print(f"np.sin:    {y_numpy}")

# (b) Generate 100×100 random data; get np.sin of the whole array
np.random.seed(42)
x_data =    # YOUR CODE HERE — np.random.randn(100, 100)
result =    # YOUR CODE HERE — np.sin of x_data (one line, no loop)
print("np.sin result shape:", result.shape)

# (c) NEW — time math.sin (looped) vs np.sin (vectorised) using %%timeit
# For timing, put each approach in its own cell below ↓


In [ ]:
%%timeit
import math, numpy as np
x = np.random.randn(100, 100)
[math.sin(xi) for row in x for xi in row]


In [ ]:
%%timeit
import numpy as np
x = np.random.randn(100, 100)
np.sin(x)


## 1.4  Array Indexing and Boolean Masking
Numpy arrays are base zero and can be flexibly indexed to allow for quick access to particular slices
```python
x = np.linspace(0,9,10)   # [0,1,2,3,4,5,6,7,8,9]
start_index = 0
end_index = 6
step=2
x[start_index:end_index:step] # [0,2,4]
```
Arrays can also be indexed in reverse by using negative indices, which can be helpful for working with the end of a long array
```python
x = np.linspace(0,999,1000)   # [0,1,2,...,997,998,999]
start_index = -6
end_index = -1
step=2
x[start_index:end_index:step] # [994,996,998]
```

NumPy also allows **Boolean indexing**: applying a logical condition to an array produces a Boolean mask (same shape, values `True`/`False`). Using that mask as an index selects only the elements where the mask is `True`.

```python
x = np.arange(10)         # [0, 1, 2, ..., 9]
mask = x > 4              # [F, F, F, F, F, T, T, T, T, T]
x[mask]                   # [5, 6, 7, 8, 9]
x[x % 2 == 0]             # [0, 2, 4, 6, 8]  — all evens
```

This is often called **fancy indexing** and avoids writing explicit `for` loops, leading to much cleaner and faster code.


In [ ]:
# ── Example 1.4 — Indexing and Boolean Masking ────────────────────
import numpy as np

x = np.arange(10)      # [0,1,2,...,9]

# Slicing  [start:stop:step]
print("x[1:6:2]:", x[1:6:2])    # [1, 3, 5]
print("x[-3:]  :", x[-3:])      # [7, 8, 9]  (last 3)

# Boolean mask
mask = x > 5
print("mask:   ", mask)          # [F F F F F F T T T T]
print("x[mask]:", x[mask])       # [6 7 8 9]

# Inline condition
print("evens:  ", x[x % 2 == 0]) # [0 2 4 6 8]

# Tilde ~ inverts a boolean mask
print("odds:   ", x[~(x % 2 == 0)])  # [1 3 5 7 9]

In [ ]:
# ── Exercise 1.4 — Boolean Masking ──────────────────────────────
x = np.arange(0, 101)

# (a) Create a Boolean mask for even numbers
is_even =    # YOUR CODE HERE  (hint: x % 2 == 0)
print("First 5 elements:", x[:5])
print("First 5 mask:", is_even[:5])

# (b) Use the mask to sum all even numbers between 0 and 100
even_sum =    # YOUR CODE HERE
print(f"Sum of even numbers 0–100: {even_sum}")   # expected: 2550

# (c) Sum of all ODD numbers between 0 and 100
odd_sum =    # YOUR CODE HERE
print(f"Sum of odd  numbers 0–100: {odd_sum}")    # expected: 2500

# (d) NEW — From a 1D array of random floats, extract all values > 0.5
np.random.seed(7)
data = np.random.rand(20)
above_half =    # YOUR CODE HERE
print(f"Values > 0.5: {above_half}")
print(f"Count: {len(above_half)}")


# 2  Optimising Python

For computationally heavy physics simulations we need to go faster than pure Python. Two key strategies:

1. **NumPy vectorisation** — replace loops with array operations.
2. **Numba JIT compilation** — compile Python/NumPy code to machine code at runtime.

We'll use the **Monte Carlo estimation of π** as our running example.

### The idea
Place a unit circle inside a 2×2 square.  
Area of circle = π, area of square = 4.  
If we throw `N` random darts at the square, the fraction landing inside the circle is ≈ π/4.

$$
\frac{N_{\text{circle}}}{N_{\text{square}}} \approx \frac{\pi}{4}
\quad \Rightarrow \quad \hat{\pi} = 4 \cdot \frac{N_{\text{circle}}}{N_{\text{square}}}
$$


In [ ]:
# ── Exercise 2.1 — Baseline Monte Carlo ─────────────────────────
import numpy as np

def monte_carlo_pi(nsamples):
    """Estimate π by counting random points inside the unit circle."""
    acc = 0
    for i in range(nsamples):
        x = np.random.random()
        y = np.random.random()
        if x**2 + y**2 < 1.0:
            acc += 1
    return 4.0 * acc / nsamples

print(monte_carlo_pi(10**4))


In [ ]:
# Time the baseline
%timeit monte_carlo_pi(10**4)


In [ ]:
# Load and run line profiler to see where time is spent
%load_ext line_profiler
%lprun -f monte_carlo_pi monte_carlo_pi(10**4)


## 2.2  NumPy-Vectorised Monte Carlo

Replace the Python loop with a single call to `np.random.random` to generate all points at once.  
Use Boolean masking to count how many land inside the circle.


In [ ]:
# ── Example 2.2 — Generating all random samples at once ──────────
import numpy as np

n = 5   # small n for illustration

# Loop approach — generates one value at a time
loop_result = []
for _ in range(n):
    loop_result.append(np.random.random())
print("loop:", loop_result)

# Vectorised — generates all values in one call
vec_result = np.random.random(n)
print("vec: ", vec_result)

# Boolean mask to count conditions without a loop
vals = np.random.random(1000)
frac_inside = (vals**2 < 0.5).sum() / 1000
print(f"Fraction where x² < 0.5: {frac_inside:.3f}  (expected ≈ {0.5**0.5:.3f})")

In [ ]:
# ── Exercise 2.2 — NumPy-Vectorised Monte Carlo ──────────────────
def numpy_calculate_pi(nsamples):
    """Estimate pi using a fully vectorised NumPy approach."""
    x =    # YOUR CODE HERE — generate nsamples uniform random values
    y =    # YOUR CODE HERE
    inside =    # YOUR CODE HERE — boolean mask: x^2 + y^2 < 1
    return    # YOUR CODE HERE — 4 * (number inside) / nsamples

print(numpy_calculate_pi(10**4))


In [ ]:
# Compare timing
%timeit monte_carlo_pi(10**4)
%timeit numpy_calculate_pi(10**4)


## 2.3  Numba JIT Compilation

[Numba](https://numba.readthedocs.io/) compiles Python functions to **optimised machine code** *at the first call* (just-in-time). The first call incurs compilation overhead; subsequent calls use the compiled binary.

```python
from numba import jit

@jit(nopython=True)
def my_fast_function(x):
    ...
```

Adding `parallel=True` and using `prange` instead of `range` enables **automatic parallelisation** across CPU cores.


In [ ]:
# ── Example 2.3 — JIT compilation with Numba ──────────────────────
from numba import jit
import numpy as np

# A plain Python loop that sums an array (deliberately slow)
def slow_sum(arr):
    total = 0.0
    for x in arr:
        total += x
    return total

# The same function JIT-compiled — first call compiles, rest are fast
@jit(nopython=True)
def fast_sum(arr):
    total = 0.0
    for x in arr:
        total += x
    return total

arr = np.random.randn(100_000)

fast_sum(arr)   # warm-up (triggers compilation)

print("slow_sum:", slow_sum(arr))
print("fast_sum:", fast_sum(arr))
%timeit slow_sum(arr)   # uncomment to compare timings
%timeit fast_sum(arr)

In [ ]:
# ── Exercise 2.3 — Numba JIT ──────────────────────────────────────
from numba import jit, prange

# (a) Apply the @jit decorator to the original Monte Carlo function
@jit(nopython=True)
def jit_calculate_pi(nsamples):
    """YOUR CODE HERE — copy and decorate monte_carlo_pi."""
    pass

# Warm-up run (triggers compilation)
jit_calculate_pi(1)
print(jit_calculate_pi(10**4))

# (b) Time after warm-up
%time jit_calculate_pi(10**4)

# (c) Parallel version using prange
@jit(nopython=True, parallel=True)
def parallel_calculate_pi(nsamples):
    """YOUR CODE HERE — same algorithm but use prange."""
    pass

parallel_calculate_pi(1)   # warm-up

%timeit jit_calculate_pi(10**4)
%timeit parallel_calculate_pi(10**4)


# 3  Working with SciPy

SciPy builds on NumPy to provide algorithms for:

| Module | Purpose |
|--------|---------|
| `scipy.optimize` | Function minimisation, root finding, curve fitting |
| `scipy.integrate` | Numerical integration (quadrature, ODE solvers) |
| `scipy.stats` | Statistical distributions and tests |
| `scipy.linalg` | Linear algebra (beyond NumPy's `linalg`) |
| `scipy.signal` | Signal processing |
| `scipy.sparse` | Sparse matrix algebra |

We will cover **minimisation**, **curve fitting**, **numerical integration** (new), **ODE solving** (new), and **bootstrapping**.


## 3.1  Minimising Functions with `scipy.optimize.minimize`

`scipy.optimize.minimize(func, x0)` numerically finds the **local minimum** of `func` starting from initial guess `x0`. It returns an `OptimizeResult` object; `result.x` is the minimising argument and `result.fun` is the function value at the minimum.

**Analytic verification:** for a potential $U(x) = \frac{1}{2}kx^2 + cx + d$, the minimum is at $x^* = -c / k$.


In [ ]:
# ── Example 3.1 — scipy.optimize.minimize ────────────────────────
import numpy as np
from scipy.optimize import minimize

# Minimise f(x) = (x - 3)²  — analytic minimum at x=3
def f(x):
    return (x[0] - 3)**2

result = minimize(f, x0=[0.0])          # start from x=0
print("Minimiser result object:")
print(result)
print()
print(f"x*  = {result.x[0]:.4f}  (expected 3.0)")
print(f"f*  = {result.fun:.2e}")
print(f"Converged: {result.success}, message: {result.message}")

In [ ]:
# ── Exercise 3.1 — Minimisation ──────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

def potential_energy(x, k=0.5, c=-0.2, d=5.0):
    """Simple harmonic potential with a linear shift."""
    return 0.5 * k * x**2 + c * x + d

# (a) Analytically compute the minimum position (x_min = -c/k)
k, c, d = 0.5, -0.2, 5.0
x_min_analytic =    # YOUR CODE HERE

# (b) Numerically find the minimum with scipy
result =    # YOUR CODE HERE — minimize(potential_energy, x0=[0.5])

print(f"Analytic minimum: x = {x_min_analytic:.4f}")
print(f"Numeric  minimum: x = {result.x[0]:.4f}, U = {result.fun:.4f}")

# (c) Plot the potential and the minimum
x_vals = np.linspace(-2, 2, 400)
plt.plot(x_vals, potential_energy(x_vals), label="U(x)")
plt.scatter(result.x, result.fun, color="red", zorder=5, label=f"min x={result.x[0]:.3f}")
plt.xlabel("x")
plt.ylabel("U(x)")
plt.legend()
plt.grid()
plt.title("Harmonic Potential Energy")
plt.show()


## 3.2  Minimising N-Dimensional Functions — Rosenbrock

SciPy's minimiser works for functions of *multiple* variables. The dimensionality is inferred from the shape of the initial guess `x0`.

The [Rosenbrock function](https://en.wikipedia.org/wiki/Rosenbrock_function) is a classic optimisation benchmark: it has a narrow curved valley, making gradient-descent methods struggle.

$$
f(x,y) = (1-x)^2 + 100(y - x^2)^2
$$

Its global minimum is at $(x, y) = (1, 1)$ where $f = 0$.


In [ ]:
# ── Example 3.2 — Minimising a 2D function ───────────────────────
import numpy as np
from scipy.optimize import minimize

# f(x,y) = x² + y²  — trivial minimum at (0,0)
def bowl(params):
    x, y = params
    return x**2 + y**2

result = minimize(bowl, x0=[2.0, -1.5])
print(f"Start [2.0, -1.5] -> found minimum at {result.x}")
print(f"f* = {result.fun:.2e}")

# With the Rosenbrock function (imported from scipy):
from scipy.optimize import rosen
result_rb = minimize(rosen, [0.0, 0.0])
print(f"Rosenbrock minimum found at {result_rb.x}  (true: [1, 1])")

In [ ]:
# ── Exercise 3.2 — 2D Minimisation ──────────────────────────────
import numpy as np, matplotlib.pyplot as plt
from scipy.optimize import minimize, rosen

# Visualise the Rosenbrock function
x = np.linspace(-2, 2, 100)
X, Y = np.meshgrid(x, x)
Z = rosen([X, Y])

fig, axs = plt.subplots(1, 2, figsize=(12, 4))
axs[0].contour(X, Y, Z, levels=np.logspace(0, 3.5, 30), cmap="viridis")
axs[0].scatter(1, 1, color="red", zorder=5, label="True min (1,1)")
axs[0].set_title("Rosenbrock contour plot")
axs[0].legend()
axs[0].set_xlabel("x"); axs[0].set_ylabel("y")

# Minimise starting from (0, 0)
result =    # YOUR CODE HERE — minimize(rosen, [0, 0])
print("Minimiser found:", result.x)   # should be close to [1, 1]

axs[0].scatter(*result.x, color="orange", zorder=6, label="Numeric min")
axs[0].legend()

# Show convergence (function value vs iteration) — nfev counts function evaluations
axs[1].set_title(f"Converged in {result.nfev} function evaluations\nfinal f = {result.fun:.2e}")
axs[1].text(0.5, 0.5, f"x* = [{result.x[0]:.4f}, {result.x[1]:.4f}]",
            transform=axs[1].transAxes, ha="center", fontsize=14)
axs[1].axis("off")
plt.tight_layout()
plt.show()


## 3.3  Curve Fitting with `scipy.optimize.curve_fit`

`curve_fit(f, x, y, p0)` adjusts the parameters of model function `f` to minimise the sum of squared residuals between the data `y` and `f(x, *params)`.

It returns:
* `popt` — optimal parameters.
* `pcov` — covariance matrix; uncertainty on parameter $i$ is $\sqrt{\text{pcov}[i,i]}$.

If measurement uncertainties `sigma` are provided, the minimisation becomes a χ² fit.


In [ ]:
# ── Example 3.3 — scipy.optimize.curve_fit ───────────────────────
import numpy as np
from scipy.optimize import curve_fit

# Fit a straight line  y = m*x + b
def line(x, m, b):
    return m * x + b

np.random.seed(0)
x_data = np.linspace(0, 5, 30)
y_data = 2.5 * x_data + 1.0 + np.random.randn(30) * 0.5

popt, pcov = curve_fit(line, x_data, y_data, p0=[1, 0])
perr = np.sqrt(np.diag(pcov))
print(f"m = {popt[0]:.3f} ± {perr[0]:.3f}  (true: 2.5)")
print(f"b = {popt[1]:.3f} ± {perr[1]:.3f}  (true: 1.0)")

# popt = best-fit parameters
# pcov = covariance matrix; diagonal entries are parameter variances

In [ ]:
# ── Exercise 3.3 — Curve Fitting ─────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

# Define an exponential-decay model: f(x) = N * exp(-x*tau) + c
def model(x, N, tau, c):
    """Exponential decay with offset."""
    return    # YOUR CODE HERE

true_params = [10, 0.5, 3]
x_plot = np.linspace(0, 10, 200)

# Generate noisy data
np.random.seed(12345)
x_data = np.random.uniform(0, 10, 50)
y_data = np.random.normal(0, 0.9, 50) + model(x_data, *true_params)

# Fit
guess = [1, 1, 1]
popt, pcov =    # YOUR CODE HERE — curve_fit(model, x_data, y_data, p0=guess)
perr =    # YOUR CODE HERE — np.sqrt(np.diag(pcov))

for t, p, e in zip(true_params, popt, perr):
    print(f"True {t:.2f} → {p:.2f} ± {e:.2f}")

# Plot
plt.figure(figsize=(8, 4))
plt.plot(x_plot, model(x_plot, *true_params), label="True model", lw=2)
plt.errorbar(x_data, y_data, fmt="o", alpha=0.5, label="Noisy data")
plt.plot(x_plot, model(x_plot, *popt), "--", label="Best fit", lw=2)
plt.xlabel("x"); plt.ylabel("y")
plt.legend(); plt.grid(); plt.title("Exponential Decay Curve Fit")
plt.show()


## 3.4  Numerical Integration 

`scipy.integrate.quad(f, a, b)` computes the definite integral $\int_a^b f(x)\,dx$ using adaptive Gaussian quadrature. It returns `(value, error_estimate)`.

For common physics integrals that have no closed form (e.g. the error function, the Gaussian normalisation, or the period of an anharmonic pendulum), numerical integration is indispensable.


In [ ]:
# ── Example 3.4 — scipy.integrate.quad ───────────────────────────
import numpy as np
from scipy.integrate import quad

# Integrate f(x) = x²  from 0 to 3  -> exact answer = 9
def f(x):
    return x**2

value, error = quad(f, 0, 3)
print(f"int_0^3 x^2 dx = {value:.6f}  (exact 9.0, error {error:.2e})")

# Infinite limits work too
def gauss(x):
    return np.exp(-x**2)

val2, _ = quad(gauss, -np.inf, np.inf)
print(f"int_{{-inf}}^{{inf}} exp(-x^2) dx = {val2:.6f}  (exact sqrt(pi) = {np.sqrt(np.pi):.6f})")

# Pass extra parameters with args=
def power(x, n):
    return x**n

val3, _ = quad(power, 0, 1, args=(4,))
print(f"int_0^1 x^4 dx = {val3:.6f}  (exact 0.2)")

In [ ]:
# ── Exercise 3.4 — Numerical Integration ────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# (a) Integrate f(x) = sin(x) from 0 to π — exact answer: 2
def f(x):
    return np.sin(x)

result, err =    # YOUR CODE HERE — quad(f, 0, np.pi)
print(f"int_0^pi sin(x) dx = {result:.6f}  (error {err:.2e})")
print(f"Exact answer           = 2.000000")

# (b) Gaussian normalisation: integral of exp(-x^2/2) from -inf to inf = sqrt(2*pi)
def gauss(x):
    return np.exp(-x**2 / 2)

result_gauss, _ =    # YOUR CODE HERE — integrate over (-np.inf, np.inf)
print(f"int exp(-x^2/2) dx = {result_gauss:.6f},  sqrt(2*pi) = {np.sqrt(2*np.pi):.6f}")

# (c) NEW — Period of an anharmonic pendulum with large amplitude θ₀
#     T = 4 * sqrt(L/g) * integral from 0 to theta0 of 1/sqrt(2*(cos(theta)-cos(theta0))) dtheta
#     Compare with the small-angle approximation T₀ = 2π*sqrt(L/g)
L, g = 1.0, 9.81
theta0_deg = 80.0
theta0 = np.radians(theta0_deg)

def integrand(theta, theta0):
    return 1.0 / np.sqrt(2 * (np.cos(theta) - np.cos(theta0)))

I, _ = quad(integrand, 0, theta0 - 1e-8, args=(theta0,))
T_exact = 4 * np.sqrt(L / g) * I
T_small = 2 * np.pi * np.sqrt(L / g)

print(f"\nPendulum period (θ₀={theta0_deg}°):")
print(f"  Large-amplitude: T = {T_exact:.4f} s")
print(f"  Small-angle approx: T₀ = {T_small:.4f} s")
print(f"  Fractional difference: {(T_exact-T_small)/T_small*100:.2f}%")


# Potential Traps with Numerical Integration

Numerical intregration, particularly older routines such as quad have a great deal of smarts built into their algorithms to evaluate the answer well and efficiently, but in some complex cases these choices can work against you. For instance if an integrand is small the total value can often lie below the default tolerance of epsabs=1.49e-8 for quad. In these cases it is more prudent to rely either only on the relative tolerance of the integrator, or to neglect efficiency and leverage more processing power in a simple method to resolve the integral

In [ ]:
import numpy as np
from scipy.integrate import quad

def f(x):
    return 1e-14 * np.exp(-x) * np.cos(50*x)

print('Default tolerance: ', quad(f, 0, 50)[0])
print('Only relative tolerance: ', quad(f, 0, 50, epsabs=0)[0])
print('Inefficient Boxcar integration: ', np.sum(f(np.arange(0,50,0.0001))*0.0001))

## 3.5  Solving Ordinary Differential Equations  

`scipy.integrate.solve_ivp(fun, t_span, y0)` solves the initial value problem (IVP):

$$
\frac{d\mathbf{y}}{dt} = f(t, \mathbf{y}), \quad \mathbf{y}(t_0) = \mathbf{y}_0
$$

Key arguments:
* `fun` — callable `f(t, y)` returning `dy/dt`.
* `t_span` — `(t_start, t_end)`.
* `y0` — initial state vector.
* `t_eval` — optional array of times at which to store the solution.

**Physics example — damped harmonic oscillator:**

$$
m \ddot{x} + b \dot{x} + k x = 0
$$

Rewritten as a first-order system with $y_0 = x$, $y_1 = \dot{x}$:

$$
\dot{y}_0 = y_1, \quad \dot{y}_1 = -\frac{b}{m} y_1 - \frac{k}{m} y_0
$$


In [ ]:
# ── Example 3.5 — scipy.integrate.solve_ivp ─────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Solve dy/dt = -y  (exponential decay), y(0)=1  -> exact: y(t)=e^{-t}
def decay(t, y):
    return [-y[0]]          # must return a list/array, even for 1D

sol = solve_ivp(decay,
                t_span=(0, 5),   # integrate from t=0 to t=5
                y0=[1.0],        # initial condition
                t_eval=np.linspace(0, 5, 100))  # where to store output

plt.figure(figsize=(6, 3))
plt.plot(sol.t, sol.y[0], label="solve_ivp")
plt.plot(sol.t, np.exp(-sol.t), 'r--', label='exact exp(-t)')
plt.xlabel("t"); plt.ylabel("y(t)")
plt.title("Exponential Decay"); plt.legend(); plt.grid()
plt.tight_layout(); plt.show()

# sol.t  — time points
# sol.y  — shape (n_vars, n_timepoints); sol.y[0] is the first variable

In [ ]:
# ── Exercise 3.5 — Damped Harmonic Oscillator ODE ────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp

# Parameters
m, k = 1.0, 4.0          # mass and spring constant
omega0 = np.sqrt(k / m)  # natural frequency

# (a) Write the ODE right-hand side for THREE damping cases
def oscillator(t, y, b):
    """Damped harmonic oscillator: returns [dy0/dt, dy1/dt]."""
    dydt =    # YOUR CODE HERE — list [y[1], (-b/m)*y[1] - (k/m)*y[0]]
    return dydt

t_span = (0, 10)
t_eval = np.linspace(*t_span, 500)
y0 = [1.0, 0.0]   # initial position = 1, initial velocity = 0

fig, ax = plt.subplots(figsize=(9, 4))
for b, label in [(0.5, "underdamped"), (2*np.sqrt(k*m), "critically damped"), (6.0, "overdamped")]:
    sol =    # YOUR CODE HERE — solve_ivp(oscillator, t_span, y0, args=(b,), t_eval=t_eval)
    ax.plot(sol.t, sol.y[0], label=f"{label} (b={b:.1f})")

ax.set_xlabel("Time [s]")
ax.set_ylabel("Displacement x(t)")
ax.set_title("Damped Harmonic Oscillator")
ax.legend()
ax.grid()
plt.show()

# (b) Solve the logistic growth equation: dy/dt = r*y*(1 - y/K)
#     Parameters: r=0.5 (growth rate), K=100 (carrying capacity), y0=5
r, K = 0.5, 100.0
y0_log = [5.0]
t_span2 = (0, 20)
t_eval2 = np.linspace(*t_span2, 300)

def logistic(t, y):
    return    # YOUR CODE HERE

sol_log =    # YOUR CODE HERE — solve_ivp(...)

plt.figure(figsize=(8, 4))
plt.plot(sol_log.t, sol_log.y[0], label="Logistic growth")
plt.axhline(K, color="r", ls="--", label=f"Carrying capacity K={K}")
plt.xlabel("Time"); plt.ylabel("Population y(t)")
plt.title("Logistic Growth")
plt.legend(); plt.grid()
plt.show()


## 3.6  Bootstrapping — Uncertainty Estimation

**Bootstrapping** is a non-parametric method for estimating the uncertainty on fit parameters without assuming Gaussian errors:

1. Start with `n` observations.
2. Resample with replacement (draw `n` values, some may repeat).
3. Fit the model to the resampled data.
4. Repeat many times; the distribution of fit parameters captures the uncertainty.

This is especially valuable when the noise is **non-Gaussian** (e.g. Poisson-distributed counts).


In [ ]:
# ── Example 3.6 — Bootstrap concept ─────────────────────────────
import numpy as np

# Suppose we want the mean of a dataset with uncertainty
np.random.seed(1)
data = np.random.normal(loc=5.0, scale=1.0, size=30)

# Standard estimate
print(f"Mean = {data.mean():.3f} ± {data.std()/np.sqrt(len(data)):.3f}  (SEM)")

# Bootstrap: resample-with-replacement many times, recompute each time
n_boot = 1000
boot_means = np.empty(n_boot)
for i in range(n_boot):
    resample = np.random.choice(data, size=len(data), replace=True)
    boot_means[i] = resample.mean()

lo, hi = np.quantile(boot_means, [0.16, 0.84])
print(f"Bootstrap 68% CI: [{lo:.3f}, {hi:.3f}]  (mean = {boot_means.mean():.3f})")
# The bootstrap CI agrees with the SEM estimate when noise is Gaussian,
# but is more reliable when the noise distribution is unknown.

In [ ]:
# ── Exercise 3.6 — Bootstrapping ─────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

def model(x, N, tau, c):
    return N * np.exp(-x * tau) + c

true_params = [10, 0.5, 3]
np.random.seed(12345)
x_data = np.random.uniform(0, 10, 100)
y_data = np.array([np.random.poisson(max(model(xi, *true_params), 1)) for xi in x_data],
                  dtype=float)

def bootstrap(func, x, y, nboot=500, p0=None):
    """Bootstrap fit: return array of shape [nboot, n_params]."""
    samples = []
    for _ in range(nboot):
        idx =    # YOUR CODE HERE — np.random.choice(len(x), size=len(x), replace=True)
        try:
            popt, _ = curve_fit(func, x[idx], y[idx], p0=p0, maxfev=5000)
            samples.append(popt)
        except RuntimeError:
            pass
    return np.array(samples)

# Fit with curve_fit first
popt, pcov = curve_fit(model, x_data, y_data, p0=[5, 1, 1], maxfev=5000)
perr = np.sqrt(np.diag(pcov))

# Bootstrap
samples = bootstrap(model, x_data, y_data, nboot=500, p0=[5, 1, 1])

param_names = ["N", "τ", "C"]
fig, axs = plt.subplots(1, 3, figsize=(12, 4))
for i, ax in enumerate(axs):
    ax.hist(samples[:, i], bins=30, alpha=0.7)
    ax.axvline(true_params[i], color="r",  ls="-",  label="True")
    ax.axvline(popt[i],         color="C2", ls="--", label="curve_fit")
    q = np.quantile(samples[:, i], [0.16, 0.84])
    ax.axvline(q[0], color="C0", ls=":", label="16th–84th pct")
    ax.axvline(q[1], color="C0", ls=":")
    ax.set_title(param_names[i]); ax.legend(fontsize=7)
plt.suptitle("Bootstrap Parameter Distributions")
plt.tight_layout(); plt.show()


# 4  Advanced Matplotlib Visualisation  

Good scientific plots communicate results clearly and honestly. This section covers the features you need most for physics coursework and publications.

---

## Understanding `fig` and `axs`

Matplotlib's object-oriented interface is built around two key objects:

| Object | What it represents |
|--------|--------------------|
| **`fig`** (`Figure`) | The entire canvas — the "page" on which everything is drawn. Controls overall size (`figsize`), resolution (`dpi`), and acts as a container for all panels. |
| **`ax` / `axs`** (`Axes`) | A single panel — one coordinate system with its own x/y axes, title, labels, and data. This is where plotting actually happens. |

```python
fig, ax  = plt.subplots()          # 1 panel  — ax  is a single Axes object
fig, axs = plt.subplots(2, 3)      # 2×3 grid — axs is a 2-D array of Axes, shape (2,3)
```

**Accessing individual panels:**
```python
axs[0, 0].plot(x, y)          # top-left panel
axs[1, 2].set_title("label")  # bottom-right panel
```

**Why use `ax.plot(...)` instead of `plt.plot(...)`?**  
`plt.plot()` targets the *most recently active* axes — fine for one panel, but unpredictable with multiple panels. Explicitly using `ax.plot(...)` ensures you are always drawing in the intended panel.

Think of `fig` as a picture frame and each `ax` as a separate canvas inside it. `plt.subplots()` hands you both the frame and all the canvases at once.

---


In [ ]:
# ── Example 4.0 — fig and axs in action ──────────────────────────
import numpy as np
import matplotlib.pyplot as plt

# Create a figure with a 1×2 grid of panels
fig, axs = plt.subplots(1, 2, figsize=(10, 4))

# fig  -> the whole canvas (frame)
# axs  -> NumPy array of Axes objects, shape (2,)
print("type(fig):", type(fig))
print("type(axs):", type(axs), "shape:", axs.shape)

x = np.linspace(0, 2*np.pi, 200)

# Each ax is addressed by index — controls are explicit and unambiguous
axs[0].plot(x, np.sin(x), color="steelblue")
axs[0].set_title("Left panel: sin(x)")
axs[0].set_xlabel("x"); axs[0].set_ylabel("y")
axs[0].grid(True)

axs[1].plot(x, np.cos(x), color="tomato")
axs[1].set_title("Right panel: cos(x)")
axs[1].set_xlabel("x")
axs[1].grid(True)

# fig-level controls (applies to the whole canvas)
fig.suptitle("fig.suptitle: title for the entire figure", fontsize=13)
plt.tight_layout()   # prevents overlapping labels
plt.show()

In [ ]:
# ── Exercise 4.1 — Multi-Panel Figures ──────────────────────────
import numpy as np
import matplotlib.pyplot as plt

x = np.linspace(0, 2*np.pi, 300)

fig, axs = plt.subplots(2, 2, figsize=(10, 7))

# (a) Top-left: sin and cos on shared panel
axs[0, 0].plot(x, np.sin(x), label="sin(x)")
axs[0, 0].plot(x, np.cos(x), label="cos(x)")
axs[0, 0].set_title("Trigonometric functions")
axs[0, 0].legend(); axs[0, 0].grid()

# (b) Top-right: YOUR CODE HERE — plot tan(x) clipped to [-5, 5]
tan_x = np.tan(x)
tan_x = np.where(np.abs(tan_x) > 5, np.nan, tan_x)  # clip large values
axs[0, 1].plot(x, tan_x)
axs[0, 1].set_title("tan(x) (clipped)")
axs[0, 1].set_ylim(-5, 5); axs[0, 1].grid()

# (c) Bottom-left: histogram of 1000 Gaussian random numbers
np.random.seed(1)
data = np.random.randn(1000)
axs[1, 0].hist(    )   # YOUR CODE HERE — bins=30, color, alpha, etc.
axs[1, 0].set_title("Gaussian histogram")
axs[1, 0].set_xlabel("Value"); axs[1, 0].set_ylabel("Count")

# (d) Bottom-right: scatter plot with error bars
x_err = np.linspace(0, 5, 15)
y_true = 2 * x_err + 1
y_meas = y_true + np.random.randn(15) * 0.8
y_err  = 0.8 * np.ones(15)
axs[1, 1].errorbar(    )   # YOUR CODE HERE — x_err, y_meas, yerr=y_err, fmt="o", capsize=4
axs[1, 1].plot(x_err, y_true, "r--", label="True line")
axs[1, 1].set_title("Measurements with error bars")
axs[1, 1].legend(); axs[1, 1].grid()

plt.suptitle("Multi-Panel Figure Example", fontsize=14)
plt.tight_layout()
plt.show()


In [ ]:
# ── Exercise 4.2 — 2D Plots (imshow, contourf, scatter coloured) ─
import numpy as np
import matplotlib.pyplot as plt

# (a) 2D Gaussian: z = exp(-(x²+y²)/2σ²)
x = np.linspace(-3, 3, 200)
X, Y = np.meshgrid(x, x)
sigma = 1.0
Z = np.exp(-(X**2 + Y**2) / (2 * sigma**2))

fig, axs = plt.subplots(1, 3, figsize=(14, 4))

im = axs[0].imshow(Z, extent=[-3, 3, -3, 3], origin="lower", cmap="hot")
axs[0].set_title("imshow — 2D Gaussian")
plt.colorbar(im, ax=axs[0])

cs = axs[1].contourf(X, Y, Z, levels=20, cmap="viridis")
axs[1].set_title("contourf — 2D Gaussian")
plt.colorbar(cs, ax=axs[1])

# (b) Coloured scatter plot: colour each point by its distance from origin
np.random.seed(42)
px = np.random.randn(300)
py = np.random.randn(300)
colours = np.sqrt(px**2 + py**2)
sc = axs[2].scatter(    )   # YOUR CODE HERE — px, py, c=colours, cmap="plasma", alpha=0.7
plt.colorbar(sc, ax=axs[2], label="Distance from origin")
axs[2].set_title("Coloured scatter")

plt.tight_layout()
plt.show()


# 5  Assignment — Optimise the Gaussian-with-Background Model

In this assignment you will use **NumPy, Numba, and SciPy** to fit a physically-motivated model to noisy data as efficiently as possible.

### Model

$$
f(x) =
\begin{cases}
A \, e^{-\frac{(x - x_0)^2}{2\sigma^2}} + Bx + C & \text{if } x \geq C_{\text{cutoff}} \\
0 & \text{if } x < C_{\text{cutoff}}
\end{cases}
$$

Parameters: amplitude $A$, peak centre $x_0$, width $\sigma$, background slope $B$, background intercept $C$, and hard cutoff $C_{\text{cutoff}}$.

### Your tasks

1. **Understand** the unoptimised reference implementation below.
2. **Rewrite** the model using `numba.vectorize` (or `@jit`) to maximise speed.
3. **Fit** the model to the provided noisy data using `scipy.optimize.curve_fit`.
4. **Quantify** parameter uncertainties using bootstrapping.
5. **Plot** the result with error bars and residuals.


In [ ]:
# Setup — run this cell first
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt

np.random.seed(42)
x = np.linspace(0, 10, 100)

A_true, x0_true, sigma_true, B_true, C_true, cutoff_true = 5.0, 5.0, 1.0, 0.5, 2.0, 1.0
p_true = [A_true, x0_true, sigma_true, B_true, C_true, cutoff_true]

y_true  = A_true * np.exp(-((x - x0_true)**2) / (2*sigma_true**2)) + B_true*x + C_true
y_true[x < cutoff_true] = 0
y_noisy = y_true + np.random.randn(len(x))

plt.figure(figsize=(8, 4))
plt.plot(x, y_true,  label="True",     lw=2)
plt.plot(x, y_noisy, "o", ms=4, alpha=0.6, label="Observed")
plt.xlabel("Distance [mm]"); plt.ylabel("Amplitude [AU]")
plt.legend(); plt.grid(); plt.title("Assignment Data")
plt.show()


In [ ]:
# ── Reference (unoptimised) implementation ───────────────────────
def gaussian_with_background(x, A, x0, sigma, B, C, C_cutoff):
    result = []
    for xi in x:
        if xi < C_cutoff:
            result.append(0.0)
        else:
            result.append(A * np.exp(-((xi - x0)**2) / (2*sigma**2)) + B*xi + C)
    return np.array(result)

guess = [1, 5, 1, 0.1, 1, 2]
%timeit opt.curve_fit(gaussian_with_background, x, y_noisy, p0=guess)


In [ ]:
# ── Task 1: Optimised Numba implementation ────────────────────────
# YOUR CODE HERE
# Use @jit or numba.vectorize to rewrite gaussian_with_background
# Time it vs the reference above

from numba import jit

@jit(nopython=True)
def gaussian_with_background_numba(x, A, x0, sigma, B, C, C_cutoff):
    # YOUR CODE HERE
    pass

# Warm up
gaussian_with_background_numba(x, *guess)

%timeit opt.curve_fit(gaussian_with_background_numba, x, y_noisy, p0=guess)


In [ ]:
# ── Task 2: Fit and visualise ──────────────────────────────────────
guess = [1, 5, 1, 0.1, 1, 2]
popt, pcov = opt.curve_fit(    )   # YOUR CODE HERE
perr = np.sqrt(np.diag(pcov))

print("\nFitted parameters:")
param_names = ["A", "x0", "σ", "B", "C", "C_cutoff"]
for name, t, p, e in zip(param_names, p_true, popt, perr):
    print(f"  {name}: true={t:.2f}, fit={p:.2f} ± {e:.2f}")

# Plot fit vs data with residuals
fig, axs = plt.subplots(2, 1, figsize=(9, 6), sharex=True)
axs[0].plot(x, y_true,  label="True model", lw=2)
axs[0].plot(x, y_noisy, "o", ms=4, alpha=0.6, label="Observed")
axs[0].plot(x, gaussian_with_background_numba(x, *popt), "--", label="Best fit", lw=2)
axs[0].set_ylabel("Amplitude")
axs[0].legend(); axs[0].grid()

residuals = y_noisy - gaussian_with_background_numba(x, *popt)
axs[1].plot(x, residuals, "o", ms=4)
axs[1].axhline(0, color="k", lw=0.8)
axs[1].set_xlabel("x [mm]"); axs[1].set_ylabel("Residuals")
axs[1].grid()
plt.tight_layout(); plt.show()


## 5.1  Bonus — Bootstrap Uncertainties

Use the `bootstrap` function from §3.6 to estimate parameter uncertainties and check whether `curve_fit`'s Gaussian error estimates are appropriate.


In [ ]:
# ── Bonus: Bootstrap ─────────────────────────────────────────────
# YOUR CODE HERE
# Run bootstrap(gaussian_with_background_numba, x, y_noisy, nboot=500, p0=guess)
# and plot the marginal distributions of all 6 parameters.
